# Lecture 26 - Bins and Windows
### EES 3350/5350 · Python in Earth Science · Friday, November 6, 2026 

Two more patterns you'll use all the time, on the same March 2021 flood data as Wednesday:
1. Bin, then summarize (`pd.cut` + `groupby`): sort every value into a class, like "low", "high", "flood", then count or average each class
2. **Moving windows** (`.rolling()`): smooth a noisy record by averaging each point with its neighbors.

Neither one is really about time. You can bin earthquakes by magnitude, rocks by grain size, ora  glacier by elevation band. You can run a moving window down an ice core (by depth) or along a river (by distance). We'll use time today because it's the data we have, but watch for the general idea.

**Learning objectives**
- sort values into classes with `pd.cut`, using your own edges and labels
- count how many values fall in each class
- smooth a record with `.rolling(N).mean()` and explain why the first values are `NaN`
- see how the window size changes what you see.


---
## Part 0 : set up

We need to load and clean the three gauges' data files exactly the way we did on Wednesday, all at once:

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

In [ ]:
folder = Path("../../course-materials/week11/usgs")
colors = {"Cumberland": "#0072B2", "Harpeth": "#E69F00", "Mill Creek": "#009E73"}

pieces = []
for path in sorted(folder.glob("*_daily.csv")):
    pieces.append(pd.read_csv(path))

daily = pd.concat(pieces, ignore_index=True)
daily = daily[["monitoring_location_id", "time", "value"]]
daily["time"] = pd.to_datetime(daily["time"])
daily = daily.sort_values(["monitoring_location_id", "time", "value"], ignore_index=True)

names = pd.DataFrame({
    "monitoring_location_id": ["USGS-03431500", "USGS-03434500", "USGS-03431060"],
    "short_name": ["Cumberland", "Harpeth", "Mill Creek"],
})
daily = pd.merge(daily, names, on="monitoring_location_id")

daily.head()
                          

#### Remember how all that worked?
This is the same `daily` table as earlier: 267 rows, one per gauge per day, Feb-Apr 2021, with a short name for each gauge.

---
## Part 1 : bin, then summarize

**Question we want to answer**:
<center>On how many days was each rver running <b>high</b>?</center>

"High" will mean something different for each river. 2,000 ft$^3$/s is a flood on Mill Creek and a quiet day on teh Cumberland. So let's compare each day to that river's own **typical** flow: it's **median** for February-April.


### Step 1 : each gauge's median (this should be review!)

`groupby`, then `.reset_index()` to move the names back into a column:

In [ ]:
grouped = daily.groupby("short_name")
medians = grouped["value"].median()
medians = medians.reset_index()
medians = medians.rename(columns={"value": "median_cfs"})

medians

### Part 2 : put each gauge's median on every one of its days (review)
A merge on `short_name` copies each median onto all 89 of that guague's days, just like the short names on Wednesday. Then divide:

In [ ]:
daily = pd.merge(daily, medians, on="short_name")
daily["times_median"] = daily["value"] / daily["median_cfs"]

daily[["short_name", "time", "value", "median_cfs", "times_median"]].head()

#### Let's confirm we know what just happened.
`times_median` says how many times its usual flow each river was caring that day. 
- `1` means a typical day
- `0.5` means half the usual flow
- `10` means ten times the usual flow

Now all three rivers are on the same scale.

### Step 3 : sort every day into a class with `pd.cut`
`pd.cut` takes a column of numbers and a list of **edges**, and says which **bin** (interval) each number flass in `labels=` fgives each bin a name. You need **one more edge than labels**, because each bin sits between two edges.

In [ ]:
edges = [0, 1, 2, 5, 10, float("inf")] # the float("inf") sets the final bin edge to go to infinity
labels = ["below median", "1-2x", "2-5x", "5-10x", "over 10x"]

daily["flow_class"] = pd.cut(daily["times_median"], bins=edges, labels=labels)

daily[["short_name", "time", "times_median", "flow_class"]].sample(6, random_state=1) # remember that .sample picks 6 rows at random and random_state=1 makes sure its the same every time

Okay. Here's how those 6 edges make 5 bins:
|**bin**|**from edge...**|**...to edge**|**label**|
|:--|:--|:--|:--|
|1|0|1|`below median`|
|2|1|2|`1-2x`|
|3|2|5|`2-5x`|
|4|3|10|`5-10x`|
|5|4|`float("inf")`|`over 10x`|

`float("inf")` is infinity, so the last bin catches **everything** above 10. 

A value exactly **on** an edge goes into the bin **below** it: 2.0 is in `1-2x`, not `2-5x`.

<div class="alert alert-danger">

**Common Error:** `ValueError: Bin labels must be one fewer than the number of bin edges` means your lists don't line up. With 5 labels you need 6 edges. Count them!

</div>

### Step 4 : summarize each class

"Bin, then sumarize" usually means **count** each bin. For one gauge, `.value_counts(sort=False)` counts each class and keeps them in order from low to high:

In [ ]:
mill = daily[daily["short_name"] == "Mill Creek"]

mill["flow_class"].value_counts(sort=False)

#### What just happened

Of Mill Creek's 89 days, about half (45) were below its median (that's what a median means!), and only 5 were more than 10 times it.

<div class="alert alert-info">

### Exercise 1 : how many high-water days did each river have?

The loop makes one bar chart per river.

1. keep only one river's rows
2. count the days in each class, in order
3. draw the bars
    
</div>

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(11, 3.5), sharey=True)

for ax, name in zip(axes, ["Mill Creek", "Harpeth", "Cumberland"]):
    one = daily[daily["short_name"] == ____ ]
    counts = one["___"].value_counts(sort=False)

    ax.bar(counts.index, counts.____, color=colors[name])
    ax.set_title(name)
    ax.tick_params(axis="x", labelrotation=45)

axes[0].set_ylabel("number of days")
fig.tight_layout();

---
## Part 2 : moving windows with `.rolling()`

Daily discharge is **noisy**: it jumps around from day to day. Sometimes youw ant to see the bigger pattern underneath. A **moving average** replaces each day with the average of a **window** of days around it.

For a 7-day window, the value of April 7 is the mean of April 1-7. Then the window slides forward one day: April 8 is the mean of April 2-8. Andd so on:

|**day**|**window (7 days ending here)**|
|:--|:--|
|April 7|April 1, 2, 3, 4, 5, 6, **7**|
|April 8|April 2, 3, 4, 5, 6, 7, **8**|
|April 9|April 3, 4, 5, 6, 7, 8, **9**|

`.rolling(7)` sets up the windows, and `mean()` averages each one. Let's try it on Mill Creek:


In [ ]:
mill = daily[daily["short_name"] == "Mill Creek"].copy()

mill["rolling_7"] = mill["value"].rolling(7).mean()

mill[["time", "value", "rolling_7"]].head(9)

The first **6** values are `NaN`. On February 1, there aren't 7 days yet to average. The first full window ends on February 7, so that's the first `rolling_7` value: the mean of February 1-7. (We used `.copy()` because we're adding a column to a piece of `daily`.)

### Figure : raw vs. smoothed

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4))

ax.plot(mill["time"], mill["value"], color="lightgray", label="daily mean")
ax.plot(mill["time"], mill["rolling_7"], color=colors["Mill Creek"], linewidth=2, label="7-day moving average")

ax.set_yscale("log")
ax.set_ylabel("discharge (ft$^3$/s)")
ax.set_title("Mill Creek, February-April 2021")
ax.legend();

<div class="alert alert-warning">

### Question 1

Look at the flood on March 28. The daily values reach 10,700 ft³/s. How high does the 7-day average get? And does the smoothed peak happen on the same day, or later?

</div>

> your answer here

<details>
<summary style="color:green;"><strong>Answer</strong></summary>

The 7-day average only reaches about 2,700 ft³/s, and it peaks on March 31, three days **after** the real peak. Averaging spreads one huge day across a week, so the peak gets **lower and later**. That's the trade-off: smoothing shows the overall pattern (wet March, drying April) but hides short events. A moving average is great for trends, and terrible for finding a flood peak.

</details>

### Want it to be centered?

`center=7` will make it so that each day sits in the **middle** of its window, with 3 days before and 3 after

|**day**|**default window (ends here)**|**`center=True` window**|
|:--|:--|:--|
|March 28|March 22-28|March 25-31|


In [ ]:
mill["rolling_7_centered"] = mill["value"].rolling(7, center=True).mean()

fig, ax = plt.subplots(figsize=(9, 4))

ax.plot(mill["time"], mill["value"], color="lightgray", label="daily mean")
ax.plot(mill["time"], mill["rolling_7"], color="gray", linestyle="--",
        label="7-day, window ends on each day")
ax.plot(mill["time"], mill["rolling_7_centered"], color=colors["Mill Creek"], linewidth=2,
        label="7-day, centered")

ax.set_yscale("log")
ax.set_ylabel("discharge (ft$^3$/s)")
ax.legend();

Now the smoothed peak lands on **March 28**, the same day as the real peak. It's still **lower** (about 2,700 instead of 10,700 ft$^3$/s), bevause averaging always flattens a peak. But it's no longer late.

**The cost**: the `NaN`s move. Instead of 6 `NaN`s at the start, you get 3 at the start and 3 at the end, because the last 3 days don't have 3 days after them yet.

<div class="alert alert-info">

### Exercise 2 : how big a window?

Try three window sizes on Mill Creek, using `center=True`.

1. the three window sizes; **3, 7 and 15** days.
2. the rolling mean, with that window, centered.
    
</div>

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(mill["time"], mill["value"], color="lightgray", label="daily mean")

for window in [____, ____, ____]:
    smooth = mill["value"].rolling(____, center=____).mean()
    ax.plot(mill["time"], smooth, linewidth=2, label=f"{window}-day")

ax.set_yscale("log")
ax.set_ylabel("discharge (ft$^3$/s)")
ax.legend();

The **bigger** the window, the smoother the line, and the more the storms get flattened: the 3-day line peaks at about 4,700 ft³/s, the 7-day at about 2,700, and the 15-day at only about 1,500, so it barely shows the March 28 flood at all. There's no "right" window: choose the one that matches the pattern you're asking about (a storm lasts days; a season lasts months), and **say which window you used**, just like saying which years a rate is for.


<div class="alert alert-success">

**Tip:** `.rolling()` only cares about **order**, not dates. Down an ice core, it averages neighboring layers (by depth). Along a river profile, it smooths elevation (by distance). Just make sure the rows are **sorted** along that axis first, because the window is made of the rows next to each other.

</div>

### `diff()` is the close cousin of `.rolling()`

Instead of averaging neighbors, it **subtracts** each value from the one before it. On daily data, that's the chagne form one day to the next: the **rate of rise** (ft$^3$/s per day)

#### Let's make a column with the difference
We can then see on which day it rose the most and by how much in Mill Creek:

In [ ]:
mill["change"] = mill["value"].diff()
biggest = mill[mill["change"] == mill["change"].max()]
print(f"{mill["short_name"][0]}")
print(biggest[["time", "value", "change"]])

...and in the Cumberland...

In [ ]:
cumb = daily[daily["short_name"] == "Cumberland"].copy()
cumb["change"] = cumb["value"].diff()
biggest = cumb[cumb["change"] == cumb["change"].max()]
print(f"{cumb["short_name"][89]}")
print(biggest[["time", "value", "change"]])

and now we can see how different those two are, as a **fraction of their median flows** to answer the question: which river rose faster, relative to its usual flow:

In [ ]:
mill_rise = mill["change"].max() / mill["median_cfs"].iloc[0]
cumb_rise = cumb["change"].max() / cumb["median_cfs"].iloc[0]
print(f"Mill Creek: {mill_rise:.0f} times its median in one day")
print(f"Cumberland: {cumb_rise:.1f} times its median in one day")

---
## A useful table

|**you want**|**write**|
|:--|:--|
|sort numbers into classes|`pd.cut(df["COL"], bins=edges, labels=labels)`|
|edges and labels|one more edge than labels; `float("inf")` for "and above"|
|count each class, in order|`df["CLASS"].value_counts(sort=False)`|
|a moving average|`df["COL"].rolling(N).mean()`|
|a centered moving average|`df["COL"].rolling(N, center=True).mean()`|
|change from the row before|`df["COL"].diff()`|

**Remember:** a moving window and `.diff()` use the rows **next to each other**, so sort first.